In [29]:
from statistics import LinearRegression

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import Logistic

In [30]:
df = pd.read_csv('train.txt', sep=';',header = None, names = ['text', 'emotion'])

In [31]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [32]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [33]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emo in unique_emotions:
    emotion_numbers[emo] = i
    i +=1
df['emotion'] = df['emotion'].map(emotion_numbers)
unique_emotions
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [34]:
df['text'] = df['text'].apply(lambda x : x.lower())

In [35]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

In [36]:
df['text'] = df['text'].apply(remove_punc)

In [37]:
def remove_num(txt):
    new = ""
    for i in txt:
        if not i.isdigit():
            new = new+i

    return new
df['text'] = df['text'].apply(remove_num)

In [38]:
def remove_emoji(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new+i
        return new
    df['text'] = df['text'].apply(remove_emoji)


In [39]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [40]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [41]:
stop_words =set(stopwords.words('english'))
print(stop_words)
len(stop_words)

{'shan', 'didn', 'ain', 'them', 'too', 'what', 'her', 'being', 'into', 'won', 'it', 'hers', 't', 'most', 'only', "couldn't", 'o', 'she', 'ourselves', 'y', 'wasn', 'the', 'couldn', "he'll", 'me', 'above', "wouldn't", 'their', "we'd", 'm', 'few', 'before', "it'll", 'having', "i'll", "mustn't", 'any', "wasn't", 'has', 'mightn', 'very', 've', 'did', 'mustn', 'under', 'he', 'will', 'or', "you'd", 'of', 'themselves', 'our', "that'll", "she'll", "they've", 'you', 'own', 'but', 'was', "hadn't", "doesn't", 'in', 'then', 'to', 'at', "it's", 'through', 'does', 'no', "aren't", 'once', 'had', 'yourselves', 'an', 'my', 'those', 'your', 'd', 'not', "haven't", 'and', "we've", 'his', 'for', 'where', 'wouldn', 'doesn', "they're", 'herself', "i'm", 'theirs', 'these', "won't", 'over', 'there', 'we', "shouldn't", "mightn't", 'this', 'they', 'nor', 'off', 'because', 'aren', 'can', 'from', 'shouldn', "hasn't", 'up', "isn't", 'myself', 'by', 'that', 'doing', "they'd", "we'll", "don't", 'between', 'am', "she's

198

In [42]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [43]:
def remove_stpwrds(txt):
    txt = txt.lower()
    words = word_tokenize(txt)
    cleaned = []
    for word in words:
        if word not in stop_words:
            cleaned.append(word)

    return ' '.join(cleaned)
df['text'] = df['text'].apply(remove_stpwrds)
df['text'].head()

0                                didnt feel humiliated
1    go feeling hopeless damned hopeful around some...
2            im grabbing minute post feel greedy wrong
3    ever feeling nostalgic fireplace know still pr...
4                                      feeling grouchy
Name: text, dtype: str

In [44]:
import sklearn
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, = train_test_split(df['text'], df['emotion'], test_size = 0.2, random_state = 42)

X_train

676      refers course though cant help feeling somehow...
12113                im starting feel im suffering fatigue
7077     feel like probably would liked book little bit...
13005                                  really feel awkward
12123    im feeling little grumpy today lame weather te...
                               ...                        
13418    love leave reader feeling confused slightly de...
5390                                         feel delicate
860                          starting feel little stressed
15795             feel stressed tired worn shape neglected
7270         feel someone rude wrongly done something lose
Name: text, Length: 12800, dtype: str

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


In [46]:
bow_vectorizer = CountVectorizer()         # bag of words

In [47]:
x_train_bow = bow_vectorizer.fit_transform(X_train)
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [48]:
x_test_bow = bow_vectorizer.transform(X_test)

In [49]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()
nb_model.fit(x_train_bow, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13359)","[[1.,1.,0.,...,0.,0.,1.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,1.,0.], [0.,0.,0.,...,0.,0.,0.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,1.,...,1.,2.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13359)","[[-10.07,-10.07,-10.76,...,-10.76,-10.76,-10.07], [ -9.6 ,-10.29,-10.29,...,-10.29,-10.29,-10.29], [-10.06,-10.06,-10.06,...,-10.06, -9.36,-10.06], [ -9.79, -9.79, -9.79,..., -9.79, -9.79, -9.79], [ -9.53,-10.22,-10.22,...,-10.22,-10.22,-10.22], [-10.91,-10.91,-10.22,...,-10.22, -9.81,-10.91]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13359


In [50]:
pred_nb = nb_model.predict(x_test_bow)
print(accuracy_score(y_test, pred_nb))


0.7678125


In [51]:
tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
x_test_tfidf = tfidf_vectorizer.transform(X_test)
x_train_tfidf
x_test_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 26934 stored elements and shape (3200, 13359)>

In [59]:
#Reverse your emotion mapping dictionary so numbers map back to words

index_to_emotion = {v: k for k, v in emotion_numbers.items()}


def predict_emotion(text):

  text_lower = text.lower()

  text_vectorized = tfidf_vectorizer.transform([text_lower])


  pred_label = tfidf_model.predict(text_vectorized)[0]

  predicted_emotion = index_to_emotion[pred_label]

  return predicted_emotion



sample_text = input("enter the text here:- ")
print(f"Text: '{sample_text}'")
print(f"Predicted Emotion: {predict_emotion(sample_text)}")

Text: 'hi i am alone and crying here'
Predicted Emotion: sadness


In [52]:
nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)
nb2_pred = nb2_model.predict(x_test_tfidf)
print(accuracy_score(y_test, nb2_pred))

0.6609375


In [54]:
from sklearn.linear_model import LogisticRegression
tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(x_train_tfidf, y_train)
tfidf_pred = tfidf_model.predict(x_test_tfidf)
print(accuracy_score(y_test,tfidf_pred))

0.8615625


In [56]:
import pandas as pd

# comparison dataframe
comparison_df = pd.DataFrame({
    'Model': [
        'Naive Bayes (Bag of Words)',
        'Naive Bayes (TF-IDF)',
        'Logistic Regression (TF-IDF)',
    ],
    'Accuracy': [0.7678, 0.6609, 0.8615],
})

print(comparison_df.sort_values(by='Accuracy', ascending=False))

                          Model  Accuracy
2  Logistic Regression (TF-IDF)    0.8615
0    Naive Bayes (Bag of Words)    0.7678
1          Naive Bayes (TF-IDF)    0.6609
